# LeetCode 2283 — Check if Number Has Equal Digit Count and Digit Value bằng Python

Link đề: https://leetcode.com/problems/check-if-number-has-equal-digit-count-and-digit-value/ · Độ khó: **Easy** · Chủ đề: String, Hash Table, Counting

Skeleton LeetCode:

```python
class Solution:
    def digitCount(self, num: str) -> bool:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: **chữ số ở vị trí `i`** cho biết **chữ số `i` xuất hiện bao nhiêu lần** trong chuỗi, và cách chuyển ký tự số thành số nguyên bằng `int(ch)`.
2. Biết đếm số lần xuất hiện bằng `str.count`, `collections.Counter` và **mảng đếm 10 ô** (counting array).
3. Biết dùng `enumerate`, `all(...)` và `sum(...)` để viết điều kiện gọn mà vẫn đúng.
4. Viết 5 cách: đếm bằng `count` cho từng vị trí, `Counter`, mảng đếm 10 ô, kiểm tra tổng chữ số trước, và so sánh hai mảng bằng `==`.
5. Trace, test ngẫu nhiên đối chiếu, và tránh lỗi: nhầm "chỉ số" với "giá trị", dùng `Counter` với khóa `int` trong khi chuỗi có khóa ký tự, quên rằng `num` dài `n` chỉ có chữ số `0..n-1` được kiểm tra.

## 2. Đề bài & ví dụ

Cho chuỗi `num` chỉ gồm các chữ số, đánh chỉ số từ `0`. Trả về `True` nếu **với mọi** chỉ số `i` (`0 <= i < n`), chữ số `i` xuất hiện **đúng `num[i]` lần** trong `num`; ngược lại trả về `False`.

| Ví dụ | num | Output | Giải thích |
|---|---|---|---|
| 1 | `"1210"` | `True` | `num[0]='1'`: chữ số `0` xuất hiện 1 lần; `num[1]='2'`: chữ số `1` xuất hiện 2 lần; `num[2]='1'`: chữ số `2` xuất hiện 1 lần; `num[3]='0'`: chữ số `3` xuất hiện 0 lần |
| 2 | `"030"` | `False` | `num[0]='0'` nhưng chữ số `0` xuất hiện 2 lần; `num[1]='3'` nhưng chữ số `1` xuất hiện 0 lần; `num[2]='0'` nhưng chữ số `2` xuất hiện 0 lần |

Constraints: `1 <= n == len(num) <= 10`; `num` chỉ gồm chữ số.

Chú ý: ở ví dụ 1, chữ số `1` xuất hiện **2 lần** ở các vị trí 0 và 2 (`'1','2','1','0'` có hai chữ `1`), đúng bằng `num[1] = 2`.

In [1]:
examples = [("1210", True), ("030", False)]
for num, expected in examples:
    rows = []
    for i, ch in enumerate(num):
        rows.append((i, int(ch), num.count(str(i))))
    print(f"num={num!r}: (i, num[i], số lần chữ số i xuất hiện) = {rows} -> mong đợi {expected}")

num='1210': (i, num[i], số lần chữ số i xuất hiện) = [(0, 1, 1), (1, 2, 2), (2, 1, 1), (3, 0, 0)] -> mong đợi True
num='030': (i, num[i], số lần chữ số i xuất hiện) = [(0, 0, 2), (1, 3, 0), (2, 0, 0)] -> mong đợi False


## 3. Kiến thức nền

### 3.1. Ký tự số và `int(ch)`

Mỗi phần tử của chuỗi `num` là một **ký tự** (chuỗi dài 1). `num[1]` là `'2'`, **không phải** số `2`. Muốn so sánh với một số phải đổi: `int('2') == 2`. Ngược lại `str(2) == '2'` đổi số thành ký tự để dùng với `count`.

In [2]:
num = "1210"
print(repr(num[1]), type(num[1]))
print(int(num[1]), type(int(num[1])))
print(num[1] == 2, int(num[1]) == 2)       # False, True: kiểu khác nhau
print(str(1), num.count(str(1)))

'2' <class 'str'>
2 <class 'int'>
False True
1 2


### 3.2. `str.count(sub)` — đếm số lần xuất hiện

`s.count(x)` đếm số lần chuỗi con `x` xuất hiện (không chồng lặp). Ở đây `x` là một ký tự số, mỗi lần gọi quét hết chuỗi: `O(n)`.

In [3]:
s = "1210"
print(s.count("1"), s.count("0"), s.count("3"))
print("aaaa".count("aa"))                  # 2: đếm không chồng lặp (khác cách nghĩ 'ký tự')

2 1 0
2


### 3.3. `collections.Counter` — bộ đếm

`Counter(s)` đếm mọi ký tự một lần, trả về từ điển `{ký tự: số lần}`. Khóa không có thì trả `0` chứ không báo lỗi — rất tiện cho bài này khi chữ số `i` không xuất hiện.

In [4]:
from collections import Counter

cnt = Counter("1210")
print(cnt)
print(cnt["1"], cnt["0"], cnt["9"])        # '9' không có -> 0
print(cnt[1])                               # khóa int khác khóa '1' -> 0: lỗi hay gặp

Counter({'1': 2, '2': 1, '0': 1})
2 1 0
0


### 3.4. Mảng đếm 10 ô (counting array)

Chữ số chỉ có 10 giá trị `0..9`, nên ta dùng list `cnt = [0] * 10`, và mỗi chữ số `d` làm `cnt[d] += 1`. Truy cập theo chỉ số nhanh và không cần băm.

In [5]:
num = "1210"
cnt = [0] * 10
for ch in num:
    cnt[int(ch)] += 1
print(cnt)                                  # cnt[d] = số lần chữ số d

[1, 2, 1, 0, 0, 0, 0, 0, 0, 0]


### 3.5. `enumerate`, `all`, `sum`

- `enumerate(num)` cho cặp `(i, ch)`.
- `all(cond for ...)` đúng khi **mọi** điều kiện đúng, dừng sớm ngay khi gặp một điều kiện sai.
- `sum(int(c) for c in num)` tính tổng các chữ số — hữu ích để kiểm tra nhanh (xem Cách 4).

In [6]:
num = "1210"
print(list(enumerate(num)))
print(all(int(ch) >= 0 for ch in num))
print(all([]))                              # True: rỗng thì mệnh đề 'với mọi' hiển nhiên đúng
print(sum(int(c) for c in num))

[(0, '1'), (1, '2'), (2, '1'), (3, '0')]
True
True
4


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đọc đề như một **bảng kê khai**: ô `i` của chuỗi nói "tôi có `num[i]` chữ số `i`". Với `num = "1210"`:

- ô 0 nói "có **1** chữ số 0" → thật: có đúng một số 0 (ở cuối). Đúng.
- ô 1 nói "có **2** chữ số 1" → thật: hai số 1 (vị trí 0 và 2). Đúng.
- ô 2 nói "có **1** chữ số 2" → thật. Đúng.
- ô 3 nói "có **0** chữ số 3" → thật. Đúng.

Tất cả đúng nên trả `True`. Với `"030"`: ô 0 nói "có 0 chữ số 0" nhưng thực tế có hai số 0 → sai ngay.

Vậy bài chỉ là **đối chiếu** hai thứ cho mọi `i`: giá trị khai báo `int(num[i])` và số lần thật `count(i)`.

In [7]:
num = "1210"
declared = [int(ch) for ch in num]
actual = [num.count(str(i)) for i in range(len(num))]
print("khai báo:", declared)
print("thực tế :", actual)
print("bằng nhau?", declared == actual)

khai báo: [1, 2, 1, 0]
thực tế : [1, 2, 1, 0]
bằng nhau? True


## 5. Cách 1 — Brute Force / baseline: `count` cho từng vị trí

**Ý tưởng:** với mỗi `i`, gọi `num.count(str(i))` rồi so với `int(num[i])`. Gặp một vị trí sai thì trả `False`.

**Độ phức tạp:** `n` lần đếm, mỗi lần `O(n)` nên `O(n^2)`. Với `n <= 10` thì chỉ là vài chục thao tác; **nút thắt** về lý thuyết là ta quét lại chuỗi `n` lần trong khi một lượt đếm là đủ.

In [8]:
class SolutionBrute:
    def digitCount(self, num: str) -> bool:
        for i in range(len(num)):
            if num.count(str(i)) != int(num[i]):
                return False
        return True

for s in ("1210", "030", "21200", "2020"):
    print(s, "->", SolutionBrute().digitCount(s))

1210 -> True
030 -> False
21200 -> True
2020 -> True


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** thay vì quét chuỗi `n` lần, **đếm tất cả chữ số trong một lượt** để có bảng `cnt[0..9]`, rồi so `cnt[i]` với `int(num[i])` cho `i = 0..n-1`.

**Invariant:** sau lượt đếm, `cnt[d]` bằng số lần chữ số `d` xuất hiện trong `num`.

**Quy tắc:** nếu có `i` với `cnt[i] != int(num[i])` thì `False`, còn lại `True`. Vì `n <= 10`, chỉ số `i` luôn nằm trong `0..9` nên mảng 10 ô là đủ.

Ý bổ sung: tổng `int(num[i])` phải bằng `n` (vì mỗi vị trí của chuỗi là một chữ số nào đó và được đếm đúng một lần). Đây là **điều kiện cần** dùng để loại nhanh (Cách 4).

## 7. Các cách tiếp theo

### Cách 2 — `Counter` một lượt

Đếm bằng `Counter(num)` với khóa là **ký tự**, nên truy cập `cnt[str(i)]`.

In [9]:
from collections import Counter

class SolutionCounter:
    def digitCount(self, num: str) -> bool:
        cnt = Counter(num)
        return all(cnt[str(i)] == int(ch) for i, ch in enumerate(num))

for s in ("1210", "030", "21200"):
    print(s, "->", SolutionCounter().digitCount(s))

1210 -> True
030 -> False
21200 -> True


**Độ phức tạp Cách 2:** Time `O(n)`, Space `O(1)` (tối đa 10 khóa).

### Cách 3 — Mảng đếm 10 ô

Dùng list 10 phần tử, duyệt hai lần: một lần đếm, một lần so sánh.

In [10]:
class SolutionArray:
    def digitCount(self, num: str) -> bool:
        cnt = [0] * 10
        for ch in num:
            cnt[int(ch)] += 1
        for i, ch in enumerate(num):
            if cnt[i] != int(ch):
                return False
        return True

for s in ("1210", "030", "21200", "0"):
    print(s, "->", SolutionArray().digitCount(s))

1210 -> True
030 -> False
21200 -> True
0 -> False


**Độ phức tạp Cách 3:** Time `O(n)`, Space `O(1)` (mảng cố định 10 ô).

### Cách 4 — Loại nhanh bằng tổng chữ số rồi mới đếm

Nếu `sum(int(c)) != n` thì chắc chắn `False` (điều kiện cần). Nếu bằng thì vẫn phải đếm để xác nhận (đây là điều kiện cần chứ chưa đủ: `"030"` có tổng `3 = n` nhưng vẫn sai).

In [11]:
class SolutionSumFirst:
    def digitCount(self, num: str) -> bool:
        n = len(num)
        if sum(int(c) for c in num) != n:      # điều kiện cần
            return False
        cnt = [0] * 10
        for c in num:
            cnt[int(c)] += 1
        return all(cnt[i] == int(num[i]) for i in range(n))

for s in ("1210", "030", "21200", "99"):
    print(s, "->", SolutionSumFirst().digitCount(s))

1210 -> True
030 -> False
21200 -> True
99 -> False


**Độ phức tạp Cách 4:** Time `O(n)`, Space `O(1)`. Phần loại nhanh không đổi bậc nhưng cho thấy cách nghĩ **điều kiện cần** — hay dùng để cắt nhánh.

### Cách 5 — So sánh hai danh sách bằng `==`

Tạo `actual = [count của i]` rồi so cả danh sách với `declared`. Cách "khai báo so với thực tế" ngắn gọn.

In [12]:
from collections import Counter

class SolutionCompare:
    def digitCount(self, num: str) -> bool:
        cnt = Counter(map(int, num))
        return [cnt[i] for i in range(len(num))] == list(map(int, num))

for s in ("1210", "030", "21200", "2020"):
    print(s, "->", SolutionCompare().digitCount(s))

1210 -> True
030 -> False
21200 -> True
2020 -> True


**Độ phức tạp Cách 5:** Time `O(n)`, Space `O(n)` cho hai danh sách tạm. Ở đây `Counter(map(int, num))` dùng khóa **số nguyên**, nên truy cập `cnt[i]` với `i` là `int` (khác Cách 2).

## 8. TRACE từng bước

Trace Cách 3 với `num = "21200"` (đây là một số thỏa: chữ số `0` xuất hiện 2 lần, `1` xuất hiện 1 lần, `2` xuất hiện 2 lần, `3` và `4` xuất hiện 0 lần).

| Lượt đếm `ch` | `cnt` sau khi cập nhật |
|---|---|
| `2` | `[0,0,1,0,0,0,0,0,0,0]` |
| `1` | `[0,1,1,0,0,0,0,0,0,0]` |
| `2` | `[0,1,2,0,0,0,0,0,0,0]` |
| `0` | `[1,1,2,0,0,0,0,0,0,0]` |
| `0` | `[2,1,2,0,0,0,0,0,0,0]` |

So sánh: `cnt[0]=2` vs `num[0]=2` đúng; `cnt[1]=1` vs `1` đúng; `cnt[2]=2` vs `2` đúng; `cnt[3]=0` vs `0` đúng; `cnt[4]=0` vs `0` đúng. Trả `True`.

In [13]:
def trace_array(num):
    cnt = [0] * 10
    print(f"num={num!r}")
    for ch in num:
        cnt[int(ch)] += 1
        print(f"  đếm {ch!r}: cnt[:{len(num)}]={cnt[:len(num)]}")
    for i, ch in enumerate(num):
        ok = cnt[i] == int(ch)
        print(f"  i={i}: cnt[{i}]={cnt[i]} vs num[{i}]={int(ch)} -> {'đúng' if ok else 'SAI'}")
        if not ok:
            return False
    return True

print("kết quả:", trace_array("21200"))
print("kết quả:", trace_array("030"))

num='21200'
  đếm '2': cnt[:5]=[0, 0, 1, 0, 0]
  đếm '1': cnt[:5]=[0, 1, 1, 0, 0]
  đếm '2': cnt[:5]=[0, 1, 2, 0, 0]
  đếm '0': cnt[:5]=[1, 1, 2, 0, 0]
  đếm '0': cnt[:5]=[2, 1, 2, 0, 0]
  i=0: cnt[0]=2 vs num[0]=2 -> đúng
  i=1: cnt[1]=1 vs num[1]=1 -> đúng
  i=2: cnt[2]=2 vs num[2]=2 -> đúng
  i=3: cnt[3]=0 vs num[3]=0 -> đúng
  i=4: cnt[4]=0 vs num[4]=0 -> đúng
kết quả: True
num='030'
  đếm '0': cnt[:3]=[1, 0, 0]
  đếm '3': cnt[:3]=[1, 0, 0]
  đếm '0': cnt[:3]=[2, 0, 0]
  i=0: cnt[0]=2 vs num[0]=0 -> SAI
kết quả: False


In [14]:
def trace_brute(num):
    for i in range(len(num)):
        real = num.count(str(i))
        print(f"  i={i}: count({str(i)!r})={real}, num[{i}]={num[i]!r}", "->", "đúng" if real == int(num[i]) else "SAI, dừng")
        if real != int(num[i]):
            return False
    return True

print("kết quả:", trace_brute("1210"))

  i=0: count('0')=1, num[0]='1' -> đúng
  i=1: count('1')=2, num[1]='2' -> đúng
  i=2: count('2')=1, num[2]='1' -> đúng
  i=3: count('3')=0, num[3]='0' -> đúng
kết quả: True


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def digitCount(self, num: str) -> bool:
        cnt = [0] * 10
        for ch in num:
            cnt[int(ch)] += 1
        for i, ch in enumerate(num):
            if cnt[i] != int(ch):
                return False
        return True

print(Solution().digitCount("1210"))
print(Solution().digitCount("030"))

True
False


## 10. Vì sao thuật toán đúng

- **Đếm đúng.** Mỗi ký tự của `num` là một chữ số `d` và được cộng đúng một lần vào `cnt[d]`, nên sau lượt đếm `cnt[d]` là số lần `d` xuất hiện.
- **Đủ chỉ số.** Đề chỉ kiểm tra `i = 0..n-1` với `n <= 10`, nên `cnt[i]` luôn nằm trong mảng 10 ô.
- **Chuỗi thỏa không có chữ số `>= n`.** Giả sử `num[j] = d` với `d >= n`. Điều kiện ở ô `j` nói chữ số `j` xuất hiện `d >= n` lần; mà chuỗi chỉ dài `n` nên `d = n` và **mọi** ký tự đều là chữ số `j`, kể cả `num[j] = j`; nhưng `j < n = d`, mâu thuẫn. Vậy mọi chữ số của một chuỗi thỏa đều nhỏ hơn `n`.
- **Điều kiện cần của Cách 4.** Với chuỗi thỏa, mọi ký tự có giá trị `< n` nên `sum(cnt[0..n-1]) = n`, và `sum(cnt[i]) = sum(num[i])`. Vậy tổng các chữ số phải **đúng bằng `n`**; nếu khác thì trả `False` ngay là đúng. Điều kiện này chưa đủ (`"030"` có tổng `3 = n` nhưng sai) nên vẫn phải đếm để xác nhận.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(num)` (`<= 10`).

| Cách | Time | Space |
|---|---|---|
| 1. `count` từng vị trí | `O(n^2)` | `O(1)` |
| 2. `Counter` | `O(n)` | `O(1)` |
| 3. Mảng đếm 10 ô | `O(n)` | `O(1)` |
| 4. Tổng trước, rồi đếm | `O(n)` | `O(1)` |
| 5. So hai danh sách | `O(n)` | `O(n)` |

Vì `n <= 10` nên mọi cách đều gần như tức thì. Mảng đếm 10 ô là lựa chọn rõ ràng, nhanh và dễ nghĩ nhất.

## 12. Bộ test

Đối chiếu mọi cách trên ví dụ đề, edge case và test ngẫu nhiên có `seed`.

In [16]:
import random

solutions = [SolutionBrute(), SolutionCounter(), SolutionArray(), SolutionSumFirst(), SolutionCompare(), Solution()]

fixed = [
    ("1210", True), ("030", False),
    ("0", False),            # ô 0 nói 'có 0 chữ số 0' nhưng thực ra có một số 0
    ("1", False),            # 'có 1 chữ số 0' nhưng không có
    ("10", False),           # ô 1 nói 'có 0 chữ số 1' nhưng thực ra có một số 1
    ("2020", True),          # 0 xuất hiện 2 lần, 1 xuất hiện 0, 2 xuất hiện 2, 3 xuất hiện 0
    ("21200", True),
    ("3211000", True),       # 0:3 lần, 1:2 lần, 2:1 lần, 3:1 lần, các chữ số 4..6 vắng mặt
    ("1210000000", False),
]
for num, expected in fixed:
    got = SolutionBrute().digitCount(num)
    assert got == expected, (num, got)
    for s in solutions:
        assert s.digitCount(num) == got, (type(s).__name__, num)
print("đã qua test cố định cho", len(solutions), "cách")

đã qua test cố định cho 6 cách


In [17]:
random.seed(2283)
dem_true = 0
for _ in range(3000):
    n = random.randint(1, 10)
    num = "".join(random.choice("0123") if random.random() < 0.8 else random.choice("0123456789") for _ in range(n))
    ref = SolutionBrute().digitCount(num)
    dem_true += ref
    for s in solutions[1:]:
        assert s.digitCount(num) == ref, (type(s).__name__, num)
print("3000 test ngẫu nhiên đối chiếu: OK; số input trả True:", dem_true)

3000 test ngẫu nhiên đối chiếu: OK; số input trả True: 1


In [18]:
# liệt kê mọi 'số tự mô tả' độ dài 4 và 7 để kiểm tra chéo
from itertools import product
dem = [''.join(p) for p in product("0123", repeat=4) if Solution().digitCount(''.join(p))]
print("độ dài 4:", dem)
assert dem == ["1210", "2020"]
print("độ dài 7 ví dụ:", "3211000", Solution().digitCount("3211000"))

độ dài 4: ['1210', '2020']
độ dài 7 ví dụ: 3211000 True


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Nhầm "chỉ số" với "giá trị"

Đề nói: chữ số **`i`** xuất hiện **`num[i]`** lần. Nhiều người so `num.count(num[i])` với `i` (đảo ngược vai trò).

In [19]:
def sai_dao_vai_tro(num):
    return all(num.count(ch) == i for i, ch in enumerate(num))       # SAI

def dung(num):
    return all(num.count(str(i)) == int(ch) for i, ch in enumerate(num))

for s in ("1210", "030"):
    print(s, "sai:", sai_dao_vai_tro(s), "đúng:", dung(s))

1210 sai: False đúng: True
030 sai: False đúng: False


### Lỗi 2: Quên đổi `str` ↔ `int`

`num.count(i)` với `i` là số nguyên gây `TypeError`; `num[i] == cnt[i]` so ký tự với số luôn `False`.

In [20]:
num = "1210"
try:
    num.count(1)
except TypeError as e:
    print("TypeError:", e)
print("so ký tự với số:", num[1] == 2, " | đã đổi:", int(num[1]) == 2)

TypeError: count() argument 1 must be str, not int
so ký tự với số: False  | đã đổi: True


### Lỗi 3: Dùng `Counter` mà tra khóa sai kiểu

`Counter(num)` có khóa là chữ `'1'`. Tra `cnt[1]` (số) luôn ra `0`, nên mọi so sánh bị lệch mà **không báo lỗi**.

In [21]:
from collections import Counter

cnt = Counter("1210")
print(cnt["1"], cnt[1])                     # 2 và 0
sai = all(cnt[i] == int(ch) for i, ch in enumerate("1210"))     # tra cnt[i] với i là int
dung = all(cnt[str(i)] == int(ch) for i, ch in enumerate("1210"))
print("sai:", sai, "| đúng:", dung)

2 0
sai: False | đúng: True


### Lỗi 4: Chỉ đếm chữ số có mặt trong chuỗi, bỏ sót chữ số vắng mặt

Chữ số vắng mặt có số đếm `0`, và vị trí tương ứng phải ghi `0`. Nếu chỉ lặp qua các chữ số **có mặt** thì không bắt được `"10"` (chữ số `1` có mặt 1 lần nhưng ô 1 ghi `0`).

In [22]:
from collections import Counter

def sai_chi_chu_so_co_mat(num):
    cnt = Counter(num)
    return all(int(num[int(d)]) == c for d, c in cnt.items() if int(d) < len(num))   # SAI

print("sai '1210':", sai_chi_chu_so_co_mat("1210"), "| '10':", sai_chi_chu_so_co_mat("10"), "| đúng '10':", Solution().digitCount("10"))
print("sai '0'   :", sai_chi_chu_so_co_mat("0"), "| đúng '0':", Solution().digitCount("0"))

sai '1210': True | '10': False | đúng '10': False
sai '0'   : False | đúng '0': False


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề liên quan "số lần xuất hiện" và bảng chữ cái/chữ số nhỏ → **mảng đếm** `[0] * 10` (hoặc 26).
- Đề "mọi `i` thỏa điều kiện" → `all(...)` hoặc vòng `for` + `return False` sớm.
- `n <= 10` nên brute `O(n^2)` vẫn qua; cứ viết cách dễ đúng nhất.
- Luôn kiểm tra kiểu: ký tự số cần `int(ch)`, và `count` cần chuỗi.
- Kiểm tra bằng tay trên `"1210"` và `"030"` trước khi nộp.

## 15. Biến thể của bài

- **Đếm** (không chỉ kiểm tra) số chuỗi độ dài `n` thỏa điều kiện: tự nhiên dẫn tới bài "số tự mô tả" (self-describing numbers).
- **Chuỗi nhiều chữ số hơn 10:** `num[i]` có thể cần nhiều chữ số; phải dùng danh sách số nguyên thay vì chuỗi.
- **So khớp bất kỳ cơ số:** cơ số `b` nghĩa là `num` là danh sách độ dài `b` với phần tử trong `0..b-1`.
- **Trả về chỉ số đầu tiên sai** thay vì `True/False`.
- **Sửa ít nhất số ký tự** để biến chuỗi thành số tự mô tả (khó hơn nhiều).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `count` từng vị trí | đếm riêng cho mỗi `i` | `O(n^2)` | `O(1)` | `n` rất nhỏ, đối chiếu |
| 2. `Counter` | một lượt đếm, tra khóa ký tự | `O(n)` | `O(1)` | code ngắn, dễ đọc |
| 3. Mảng 10 ô | đếm rồi so | `O(n)` | `O(1)` | nhanh nhất, chuẩn thi |
| 4. Tổng trước | loại nhanh khi `sum != n` | `O(n)` | `O(1)` | ví dụ về điều kiện cần |
| 5. So hai danh sách | `[cnt[i]] == [num[i]]` | `O(n)` | `O(n)` | một dòng, trực quan |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, dự đoán `Solution().digitCount("2020")` và `Solution().digitCount("1200")`.

**Bài 2 (tính tay).** Với `num = "21200"`, viết ra `cnt[0..4]` rồi giải thích vì sao trả `True`.

**Bài 3 (điền TODO).** Hoàn thành `count_digits(num)` trả về mảng 10 ô đếm các chữ số (cell dưới chưa được gọi).

**Bài 4 (tự viết lại).** Viết `first_wrong_index(num)` trả về chỉ số `i` đầu tiên không thỏa, hoặc `-1` nếu đúng hết.

**Bài 5 (mini challenge).** Viết `self_describing(n)` trả về danh sách mọi chuỗi độ dài `n` (`1 <= n <= 7`) thỏa điều kiện (gợi ý: `itertools.product`).

In [23]:
# TODO Bài 3: điền vào chỗ trống, KHÔNG gọi hàm trong cell này
def count_digits_todo(num):
    cnt = [0] * 10
    # for ch in num: ...
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"2020"` có hai số 2 và hai số 0 nên chữ số 0 xuất hiện 2 lần (ô 0 = 2), chữ số 2 xuất hiện 2 lần (ô 2 = 2). `"1200"`: ô 0 nói 1 nhưng có hai số 0.
- **Mức 2 (vừa):** Bài 3: vòng `for ch in num: cnt[int(ch)] += 1`. Bài 4: dùng `enumerate` và `return i` khi `cnt[i] != int(ch)`.
- **Mức 3 (sâu):** Bài 5: với mỗi tổ hợp `p` của `product("0123456", repeat=n)` chỉ giữ những tổ hợp mà `Solution().digitCount` trả `True`. Chú ý nhiều chữ số hơn `n` bị loại tự nhiên.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
from itertools import product

# Bài 1 và 2
assert Solution().digitCount("2020") is True
assert Solution().digitCount("1200") is False
cnt = [0] * 10
for ch in "21200":
    cnt[int(ch)] += 1
assert cnt[:5] == [2, 1, 2, 0, 0]

# Bài 3
def count_digits(num):
    cnt = [0] * 10
    for ch in num:
        cnt[int(ch)] += 1
    return cnt

# Bài 4
def first_wrong_index(num):
    cnt = count_digits(num)
    for i, ch in enumerate(num):
        if cnt[i] != int(ch):
            return i
    return -1

# Bài 5
def self_describing(n):
    return [''.join(p) for p in product("0123456", repeat=n) if Solution().digitCount(''.join(p))]

assert count_digits("1210")[:4] == [1, 2, 1, 0]
assert first_wrong_index("030") == 0
assert first_wrong_index("1210") == -1
assert first_wrong_index("10") == 1
assert self_describing(4) == ["1210", "2020"]
assert self_describing(1) == [] and self_describing(2) == []
assert "21200" in self_describing(5)
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm một lượt rồi so)

```python
class Solution:
    def digitCount(self, num: str) -> bool:
        cnt = [0] * 10
        for ch in num:
            cnt[int(ch)] += 1
        return all(cnt[i] == int(ch) for i, ch in enumerate(num))
```

**Checklist**

- [ ] Nhớ đúng vai trò: chữ số `i` xuất hiện `num[i]` lần (chỉ số là chữ số cần đếm, giá trị là số lần)?
- [ ] Đã đổi `str` sang `int` ở đúng chỗ (`int(ch)`, `str(i)`)?
- [ ] Mảng đếm đủ 10 ô và truy cập `cnt[i]` với `i < n <= 10`?
- [ ] Nếu dùng `Counter`, khóa là ký tự (`cnt[str(i)]`)?
- [ ] Đã thử `"1210"` (True), `"030"` (False), `"0"` (False), `"2020"` (True)?
- [ ] Signature đúng `digitCount(self, num: str) -> bool`?